In [1]:
# Cell 1: Environment setup
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"] = "1"

In [2]:
# Cell 2: Imports
import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import torchaudio
import random
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2ForSequenceClassification, Wav2Vec2FeatureExtractor

In [3]:
# Cell 3: Check GPU
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

PyTorch version: 2.12.0.dev20260312+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 5070 Laptop GPU


In [4]:
# Cell 4: Load ASVspoof data + WhatsApp voice notes
import os

# Paths - use relative paths or environment variables
ASVSPOOF_ROOT = os.environ.get("ASVSPOOF_ROOT", r"C:\deepfake-project\data\asvspoof")
PROTOCOL_DIR  = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_cm_protocols")
TRAIN_AUDIO   = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_train", "flac")
DEV_AUDIO     = os.path.join(ASVSPOOF_ROOT, "ASVspoof2019_LA_dev", "flac")

# Path to your collected WhatsApp voice notes
WHATSAPP_VOICE_DIR = r"C:\whatsapp-voice-bot\ChrisKelleher1947.github.io\bot\collected_voice_notes"

# Verify paths exist
for path_name, path in [("ASVSPOOF_ROOT", ASVSPOOF_ROOT), ("PROTOCOL_DIR", PROTOCOL_DIR), 
                         ("TRAIN_AUDIO", TRAIN_AUDIO), ("DEV_AUDIO", DEV_AUDIO)]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path_name} does not exist: {path}")

if not os.path.exists(WHATSAPP_VOICE_DIR):
    print(f"WARNING: WhatsApp voice directory not found: {WHATSAPP_VOICE_DIR}")
    print("Proceeding with ASVspoof data only")
 
# Read train and dev protocol files
train_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.train.trn.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
dev_df = pd.read_csv(
    os.path.join(PROTOCOL_DIR, "ASVspoof2019.LA.cm.dev.trl.txt"),
    sep=" ", header=None,
    names=["speaker", "file_id", "env", "attack", "label"]
)
 
# Convert labels to integers — 0 = bonafide, 1 = spoof
train_df["label"] = train_df["label"].map({"bonafide": 0, "spoof": 1})
dev_df["label"]   = dev_df["label"].map({"bonafide": 0, "spoof": 1})
 
# Add the full file path for each audio file
train_df["path"] = train_df["file_id"].apply(lambda x: os.path.join(TRAIN_AUDIO, f"{x}.flac"))
dev_df["path"]   = dev_df["file_id"].apply(lambda x: os.path.join(DEV_AUDIO,   f"{x}.flac"))

# Add WhatsApp voice notes if directory exists
if os.path.exists(WHATSAPP_VOICE_DIR):
    whatsapp_files = [f for f in os.listdir(WHATSAPP_VOICE_DIR) if f.endswith('.ogg')]
    
    whatsapp_df = pd.DataFrame({
        "speaker": ["chris"] * len(whatsapp_files),
        "file_id": [os.path.splitext(f)[0] for f in whatsapp_files],
        "env": ["whatsapp"] * len(whatsapp_files),
        "attack": ["-"] * len(whatsapp_files),
        "label": [0] * len(whatsapp_files),  # All are bonafide (REAL)
        "path": [os.path.join(WHATSAPP_VOICE_DIR, f) for f in whatsapp_files]
    })
    
    # Merge with training data
    train_df_original_count = len(train_df)
    train_df = pd.concat([train_df, whatsapp_df], ignore_index=True)
    
    print(f"  Added {len(whatsapp_df)} WhatsApp voice notes to training set")
    print(f"  ASVspoof samples: {train_df_original_count}")
    print(f"  WhatsApp samples: {len(whatsapp_df)}")
    print(f"  Total:            {len(train_df)}")
else:
    print("No WhatsApp voice notes added (directory not found)")

print(f"\nFinal dataset:")
print(f"Training samples:   {len(train_df)}")
print(f"Dev samples:        {len(dev_df)}")
print(f"Train label split:  {train_df['label'].value_counts().to_dict()}")
print(f"Dev label split:    {dev_df['label'].value_counts().to_dict()}")

  Added 50 WhatsApp voice notes to training set
  ASVspoof samples: 25380
  WhatsApp samples: 50
  Total:            25430

Final dataset:
Training samples:   25430
Dev samples:        24844
Train label split:  {1: 22800, 0: 2630}
Dev label split:    {1: 22296, 0: 2548}


In [5]:
# Cell 5: Dataset with WhatsApp-style augmentation (FIXED FOR WINDOWS)
import os
import tempfile
import subprocess
import hashlib
import numpy as np
import torch
import random
import soundfile as sf
from torch.utils.data import Dataset

SAMPLE_RATE = 16000
MAX_SAMPLES = 64000  # 4 seconds

OPUS_CACHE_DIR = os.environ.get("OPUS_CACHE_DIR", None)
if OPUS_CACHE_DIR:
    os.makedirs(OPUS_CACHE_DIR, exist_ok=True)
    print(f"Opus compression caching enabled at: {OPUS_CACHE_DIR}")


class ASVspoofDataset(Dataset):

    def __init__(self, df, feature_extractor, augment=True):
        self.df = df
        self.feature_extractor = feature_extractor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    # =========================
    # FIXED AUDIO LOADER
    # =========================
    def load_audio(self, path):
        """Load .flac or .ogg safely on Windows"""

        if path.endswith(".ogg"):
            tmp_wav = tempfile.mktemp(suffix=".wav")

            try:
                result = subprocess.run([
                    "ffmpeg", "-y", "-loglevel", "error",
                    "-i", path,
                    "-ar", str(SAMPLE_RATE),
                    "-ac", "1",
                    tmp_wav
                ], capture_output=True, text=True)

                if result.returncode != 0:
                    raise Exception(result.stderr)

                audio, sr = sf.read(tmp_wav, dtype="float32")
                return audio

            finally:
                if os.path.exists(tmp_wav):
                    try:
                        os.remove(tmp_wav)
                    except PermissionError:
                        pass

        else:
            audio, sr = sf.read(path, dtype="float32")
            return audio

    # =========================
    # FIXED OPUS AUGMENTATION
    # =========================
    def apply_opus_compression(self, audio_tensor, sr=16000, source_path=None):

        if OPUS_CACHE_DIR and source_path:
            cache_key = hashlib.md5(source_path.encode()).hexdigest()
            cache_file = os.path.join(OPUS_CACHE_DIR, f"{cache_key}.wav")

            if os.path.exists(cache_file):
                audio, _ = sf.read(cache_file, dtype="float32")
                return torch.from_numpy(audio).unsqueeze(0)

        audio_np = audio_tensor.squeeze(0).numpy()

        tmp_in = tempfile.mktemp(suffix=".wav")
        tmp_ogg = tempfile.mktemp(suffix=".ogg")
        tmp_out = tempfile.mktemp(suffix=".wav")

        try:
            sf.write(tmp_in, audio_np, sr)

            bitrate = random.choice([16, 24, 32])

            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_in,
                "-c:a", "libopus",
                "-b:a", f"{bitrate}k",
                "-ar", str(sr),
                tmp_ogg
            ], capture_output=True, text=True)

            if result.returncode != 0:
                return audio_tensor

            result = subprocess.run([
                "ffmpeg", "-y", "-loglevel", "error",
                "-i", tmp_ogg,
                "-ar", str(sr),
                "-ac", "1",
                tmp_out
            ], capture_output=True, text=True)

            if result.returncode != 0:
                return audio_tensor

            compressed_audio, _ = sf.read(tmp_out, dtype="float32")

            if OPUS_CACHE_DIR and source_path:
                cache_key = hashlib.md5(source_path.encode()).hexdigest()
                cache_file = os.path.join(OPUS_CACHE_DIR, f"{cache_key}.wav")
                sf.write(cache_file, compressed_audio, sr)

            return torch.from_numpy(compressed_audio).unsqueeze(0)

        finally:
            for f in [tmp_in, tmp_ogg, tmp_out]:
                if f and os.path.exists(f):
                    try:
                        os.remove(f)
                    except:
                        pass

    # =========================
    # DATA FETCH
    # =========================
    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        audio = self.load_audio(row["path"])
        audio = audio.astype(np.float32)

        audio_tensor = torch.from_numpy(audio).unsqueeze(0)

        is_whatsapp = row["path"].endswith(".ogg")

        if self.augment and random.random() < 0.7:

            if not is_whatsapp and random.random() < 0.5:
                audio_tensor = self.apply_opus_compression(
                    audio_tensor, SAMPLE_RATE, row["path"]
                )

            if random.random() < 0.4:
                noise = torch.randn_like(audio_tensor) * random.uniform(0.001, 0.01)
                audio_tensor = audio_tensor + noise

            if random.random() < 0.3:
                audio_tensor = audio_tensor * random.uniform(0.7, 1.3)

        audio = audio_tensor.squeeze(0).numpy()

        if len(audio) >= MAX_SAMPLES:
            audio = audio[:MAX_SAMPLES]
        else:
            audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))

        peak = np.abs(audio).max()
        if peak > 0:
            audio = audio / peak

        inputs = self.feature_extractor(
            audio,
            sampling_rate=SAMPLE_RATE,
            return_tensors="pt",
            padding=False
        )

        return {
            "input_values": inputs["input_values"].squeeze(0).to(torch.bfloat16),
            "label": torch.tensor(row["label"], dtype=torch.long)
        }

In [6]:
# Cell 6: Load model with proper label mapping
MODEL_NAME = "facebook/wav2vec2-base"
device     = torch.device("cuda")
 
print("Loading feature extractor...")
feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(MODEL_NAME)
 
print("Loading model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    ignore_mismatched_sizes=True
)
 
# Set proper label names (this will be saved with the model)
model.config.id2label = {0: "bonafide", 1: "spoof"}
model.config.label2id = {"bonafide": 0, "spoof": 1}
 
# Freeze the CNN feature extractor
for param in model.wav2vec2.feature_extractor.parameters():
    param.requires_grad = False
 
# Freeze the bottom 6 transformer layers (half of 12)
for i in range(6):
    for param in model.wav2vec2.encoder.layers[i].parameters():
        param.requires_grad = False
 
# Move model to GPU then convert weights to bfloat16
model = model.to(device)
model = model.to(torch.bfloat16)
 
# Allow TF32
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
 
print(f"Model loaded on {device}")
 
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,}")
print(f"Frozen parameters:    {total - trainable:,} / {total:,}")
print(f"\nLabel mapping: {model.config.id2label}")

Loading feature extractor...


Loading model...


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
quantizer.codevectors        | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
project_hid.bias             | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
projector.bias               | MISSING    | 
projector.weight             | MISSING    | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda
Trainable parameters: 47,841,410 / 94,569,090
Frozen parameters:    46,727,680 / 94,569,090

Label mapping: {0: 'bonafide', 1: 'spoof'}


In [7]:
# Cell 7: Create data loaders
from torch.utils.data import WeightedRandomSampler
 
# Create dataset objects with augmentation
train_dataset = ASVspoofDataset(train_df, feature_extractor, augment=True)
dev_dataset   = ASVspoofDataset(dev_df,   feature_extractor, augment=False)  # No augmentation for validation
 
# Handle class imbalance using a weighted sampler
class_counts  = train_df["label"].value_counts().sort_index().values
class_weights = 1.0 / class_counts
sample_weights = train_df["label"].map({0: class_weights[0], 1: class_weights[1]}).values

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(train_dataset),  # Fixed: use dataset length
    replacement=False  # Fixed: no replacement to ensure all samples seen once per epoch
)
 
# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    sampler=sampler,
    num_workers=0,
    pin_memory=True  # Added for faster GPU transfer
)
 
dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)
 
print(f"Training batches:   {len(train_loader)}")
print(f"Development batches: {len(dev_loader)}")
print("✓ Data augmentation enabled for training set")
print(f"✓ Class weights: bonafide={class_weights[0]:.4f}, spoof={class_weights[1]:.4f}")

Training batches:   3179
Development batches: 3106
✓ Data augmentation enabled for training set
✓ Class weights: bonafide=0.0004, spoof=0.0000


c:\Users\tkell\anaconda3\envs\deepfake\Lib\site-packages\torch\utils\data\sampler.py:264: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  weights_tensor = torch.as_tensor(weights, dtype=torch.double)


In [8]:
# Cell 8: Optimizer and evaluation function
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from sklearn.metrics import roc_auc_score
 
# AdamW optimiser
optimiser = AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01
)
 
# Linear warmup scheduler
total_steps  = len(train_loader) * 5  # 5 epochs instead of 3
warmup_steps = int(0.1 * total_steps)
scheduler = LinearLR(
    optimiser,
    start_factor=0.1,
    end_factor=1.0,
    total_iters=warmup_steps
)
 
def evaluate(model, loader, device):
    """Run model on dev set and return loss, accuracy and ROC-AUC."""
    model.eval()
    total_loss, correct, all_labels, all_probs = 0, 0, [], []
 
    with torch.no_grad():
        for batch in loader:
            input_values = batch["input_values"].to(device)
            labels       = batch["label"].to(device)
 
            outputs = model(input_values=input_values, labels=labels)
            total_loss += outputs.loss.item()
 
            probs  = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
            preds  = probs.argmax(dim=-1)
            correct += (preds == labels).sum().item()
 
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
 
    avg_loss = total_loss / len(loader)
    accuracy = correct / len(loader.dataset)
    roc_auc  = roc_auc_score(all_labels, all_probs)
    return avg_loss, accuracy, roc_auc
 
print("Optimiser and evaluation function ready")
print(f"Total training steps: {total_steps}")
print(f"Warmup steps:         {warmup_steps}")

Optimiser and evaluation function ready
Total training steps: 15895
Warmup steps:         1589


In [9]:
# Cell 9: Training loop
from torch.amp import autocast
from sklearn.metrics import roc_auc_score
 
EPOCHS       = 5  # Increased from 3
EVAL_STEPS   = 200
SAVE_DIR     = r"C:\deepfake-project\models\wav2vec2_finetuned"
best_roc_auc = 0.0
patience     = 0
PATIENCE_MAX = 4  # Increased patience
 
os.makedirs(SAVE_DIR, exist_ok=True)
 
print("Starting training with data augmentation...")
print(f"Evaluating every {EVAL_STEPS} steps — best model saved to {SAVE_DIR}")
print("-" * 60)
 
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0
    step       = 0
 
    for batch in train_loader:
        input_values = batch["input_values"].to(device)
        labels       = batch["label"].to(device)
 
        with autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(input_values=input_values, labels=labels)
        loss = outputs.loss
 
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimiser.step()
        scheduler.step()
        optimiser.zero_grad()
 
        epoch_loss += loss.item()
        step       += 1
 
        if step % 50 == 0:
            avg_loss = epoch_loss / step
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | Loss: {avg_loss:.4f}")
 
        if step % EVAL_STEPS == 0:
            dev_loss, dev_acc, dev_roc = evaluate(model, dev_loader, device)
            print(f"\n>>> Eval @ step {step} | Loss: {dev_loss:.4f} | Acc: {dev_acc:.4f} | ROC-AUC: {dev_roc:.4f}")
 
            if dev_roc > best_roc_auc:
                best_roc_auc = dev_roc
                patience     = 0
                model.save_pretrained(SAVE_DIR)
                feature_extractor.save_pretrained(SAVE_DIR)
                print(f"    ✓ New best model saved (ROC-AUC: {best_roc_auc:.4f})")
            else:
                patience += 1
                print(f"    No improvement — patience {patience}/{PATIENCE_MAX}")
                if patience >= PATIENCE_MAX:
                    print("\nEarly stopping triggered")
                    break
 
            model.train()
 
    if patience >= PATIENCE_MAX:
        break
 
    print(f"\nEpoch {epoch+1} complete | Avg loss: {epoch_loss/len(train_loader):.4f}\n")
 
print("-" * 60)
print(f"Training complete | Best ROC-AUC: {best_roc_auc:.4f}")
print(f"Best model saved to: {SAVE_DIR}")

Starting training with data augmentation...
Evaluating every 200 steps — best model saved to C:\deepfake-project\models\wav2vec2_finetuned
------------------------------------------------------------
Epoch 1 | Step 50/3179 | Loss: 0.6937
Epoch 1 | Step 100/3179 | Loss: 0.6919
Epoch 1 | Step 150/3179 | Loss: 0.6894
Epoch 1 | Step 200/3179 | Loss: 0.6863

>>> Eval @ step 200 | Loss: 0.6176 | Acc: 0.8974 | ROC-AUC: 0.8835


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.8835)
Epoch 1 | Step 250/3179 | Loss: 0.6825
Epoch 1 | Step 300/3179 | Loss: 0.6690
Epoch 1 | Step 350/3179 | Loss: 0.6330
Epoch 1 | Step 400/3179 | Loss: 0.5923

>>> Eval @ step 400 | Loss: 0.4059 | Acc: 0.8260 | ROC-AUC: 0.9786


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9786)
Epoch 1 | Step 450/3179 | Loss: 0.5515
Epoch 1 | Step 500/3179 | Loss: 0.5145
Epoch 1 | Step 550/3179 | Loss: 0.4794
Epoch 1 | Step 600/3179 | Loss: 0.4515

>>> Eval @ step 600 | Loss: 0.6465 | Acc: 0.7885 | ROC-AUC: 0.9812


...

    ✓ New best model saved (ROC-AUC: 0.9812)
Epoch 1 | Step 650/3179 | Loss: 0.4241
Epoch 1 | Step 700/3179 | Loss: 0.4014
Epoch 1 | Step 750/3179 | Loss: 0.3798
Epoch 1 | Step 800/3179 | Loss: 0.3596

>>> Eval @ step 800 | Loss: 0.3461 | Acc: 0.8996 | ROC-AUC: 0.9939


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9939)
Epoch 1 | Step 850/3179 | Loss: 0.3439
Epoch 1 | Step 900/3179 | Loss: 0.3302
Epoch 1 | Step 950/3179 | Loss: 0.3160
Epoch 1 | Step 1000/3179 | Loss: 0.3022

>>> Eval @ step 1000 | Loss: 0.2543 | Acc: 0.9335 | ROC-AUC: 0.9932
    No improvement — patience 1/4
Epoch 1 | Step 1050/3179 | Loss: 0.2898
Epoch 1 | Step 1100/3179 | Loss: 0.2793
Epoch 1 | Step 1150/3179 | Loss: 0.2686
Epoch 1 | Step 1200/3179 | Loss: 0.2589

>>> Eval @ step 1200 | Loss: 0.4243 | Acc: 0.8966 | ROC-AUC: 0.9886
    No improvement — patience 2/4
Epoch 1 | Step 1250/3179 | Loss: 0.2490
Epoch 1 | Step 1300/3179 | Loss: 0.2416
Epoch 1 | Step 1350/3179 | Loss: 0.2329
Epoch 1 | Step 1400/3179 | Loss: 0.2248

>>> Eval @ step 1400 | Loss: 0.1558 | Acc: 0.9622 | ROC-AUC: 0.9944


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

    ✓ New best model saved (ROC-AUC: 0.9944)
Epoch 1 | Step 1450/3179 | Loss: 0.2178
Epoch 1 | Step 1500/3179 | Loss: 0.2126
Epoch 1 | Step 1550/3179 | Loss: 0.2071
Epoch 1 | Step 1600/3179 | Loss: 0.2007

>>> Eval @ step 1600 | Loss: 0.0987 | Acc: 0.9734 | ROC-AUC: 0.9928
    No improvement — patience 1/4
Epoch 1 | Step 1650/3179 | Loss: 0.1952
Epoch 1 | Step 1700/3179 | Loss: 0.1900
Epoch 1 | Step 1750/3179 | Loss: 0.1846
Epoch 1 | Step 1800/3179 | Loss: 0.1797

>>> Eval @ step 1800 | Loss: 0.2754 | Acc: 0.9339 | ROC-AUC: 0.9919
    No improvement — patience 2/4
Epoch 1 | Step 1850/3179 | Loss: 0.1750
Epoch 1 | Step 1900/3179 | Loss: 0.1704
Epoch 1 | Step 1950/3179 | Loss: 0.1661
Epoch 1 | Step 2000/3179 | Loss: 0.1620

>>> Eval @ step 2000 | Loss: 0.0768 | Acc: 0.9810 | ROC-AUC: 0.9935
    No improvement — patience 3/4
Epoch 1 | Step 2050/3179 | Loss: 0.1581
Epoch 1 | Step 2100/3179 | Loss: 0.1544
Epoch 1 | Step 2150/3179 | Loss: 0.1508
Epoch 1 | Step 2200/3179 | Loss: 0.1475

>>> E

In [10]:
# Cell 10: Test saved model
saved_model = Wav2Vec2ForSequenceClassification.from_pretrained(SAVE_DIR)
saved_extractor = Wav2Vec2FeatureExtractor.from_pretrained(SAVE_DIR)
 
saved_model = saved_model.to(device)
saved_model = saved_model.to(torch.bfloat16)
saved_model.eval()
 
print(f"\n=== Saved Model Configuration ===")
print(f"id2label: {saved_model.config.id2label}")
print(f"label2id: {saved_model.config.label2id}")
 
def predict(audio_path):
    audio, sr = sf.read(audio_path)
    audio = audio.astype(np.float32)
 
    if len(audio) >= MAX_SAMPLES:
        audio = audio[:MAX_SAMPLES]
    else:
        audio = np.pad(audio, (0, MAX_SAMPLES - len(audio)))
 
    peak = np.abs(audio).max()
    if peak > 0:
        audio = audio / peak
 
    inputs = saved_extractor(
        audio,
        sampling_rate=SAMPLE_RATE,
        return_tensors="pt",
        padding=False
    )
 
    input_values = inputs["input_values"].to(device).to(torch.bfloat16)
 
    with torch.no_grad():
        outputs = saved_model(input_values=input_values)
        probs = torch.softmax(outputs.logits, dim=-1).to(torch.float32)
 
    return {
        "bonafide": round(probs[0][0].item(), 4),
        "spoof":    round(probs[0][1].item(), 4),
        "verdict":  "REAL" if probs[0][0] > probs[0][1] else "FAKE"
    }
 
# Test on dev set
real_file = dev_df[dev_df["label"] == 0].iloc[0]["path"]
fake_file = dev_df[dev_df["label"] == 1].iloc[0]["path"]
 
print("\n=== Testing saved model ===")
print(f"\nReal audio: {os.path.basename(real_file)}")
print(predict(real_file))
 
print(f"\nFake audio: {os.path.basename(fake_file)}")
print(predict(fake_file))

Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]


=== Saved Model Configuration ===
id2label: {0: 'bonafide', 1: 'spoof'}
label2id: {'bonafide': 0, 'spoof': 1}

=== Testing saved model ===

Real audio: LA_D_1047731.flac
{'bonafide': 0.9961, 'spoof': 0.003, 'verdict': 'REAL'}

Fake audio: LA_D_1008730.flac
{'bonafide': 0.002, 'spoof': 0.9961, 'verdict': 'FAKE'}
